# Домашняя работа 3.2. Хаос в пищевой цепи
## Жёлтый уровень

Сравниваются режимы модели Hastings–Powell при $b_1=2.3$ и $b_1=3$:

$$
\dot x=x(1-x)-f_1(x)y,\qquad
\dot y=f_1(x)y-f_2(y)z-d_1y,
$$

$$
\dot z=f_2(y)z-d_2z,\qquad f_i(u)=\frac{a_i u}{1+b_i u}.
$$

Остальные параметры: $a_1=5$, $a_2=0.1$, $b_2=2$, $d_1=0.4$, $d_2=0.01$.
Начальное состояние в обоих случаях $(x,y,z)=(0.5,0.1,0.1)$.
Все переменные и время безразмерны.


## 1. Расстояние и перенормировка возмущения

Для координат выбраны постоянные масштабы $S=(1,0.5,10)$.
Они соответствуют характерным порядкам популяций: $x$ порядка единицы,
$y$ — десятых долей, $z$ — десятков. Эти значения одинаковы для обоих режимов,
всех шагов и всех временных интервалов. Расстояние между состояниями:

$$
\rho=\sqrt{(\Delta x/1)^2+(\Delta y/0.5)^2+(\Delta z/10)^2}.
$$

Одновременно интегрируются основная и близкая траектории методом RK4.
Начальное возмущение равно $\Delta\mathbf u=\varepsilon S/\sqrt3$,
поэтому его длина в выбранной норме равна $\varepsilon=10^{-7}$.
Через каждый интервал $\Delta t_r=1$ измеряется $\rho_k$ и накапливается
$\ln(\rho_k/\varepsilon)$. Затем возмущённая траектория возвращается на расстояние
$\varepsilon$ с сохранением направления:

$$
\widetilde{\mathbf u}\leftarrow\mathbf u+
\frac{\varepsilon}{\rho_k}(\widetilde{\mathbf u}-\mathbf u).
$$

Перенормировка выполняется и во время переходного процесса, чтобы направление
возмущения успело выровняться вдоль наиболее быстро растущего направления.
Логарифмы до выбранного времени $t_{\mathrm{пер}}$ не входят в среднее.
Для последующего интервала длины $T=N\Delta t_r$:

$$
\lambda_{\max}(T)=\frac{1}{T}\sum_{k=1}^{N}\ln\frac{\rho_k}{\varepsilon}.
$$


In [1]:
import numpy as np

scales = np.array([1.0, 0.5, 10.0])
parameters = np.array([2.3, 3.0])

def rhs(states):
    x, y, z = np.moveaxis(states, -1, 0)
    f1 = 5 * x / (1 + parameters[:, None] * x)
    f2 = 0.1 * y / (1 + 2 * y)
    return np.stack([x * (1 - x) - f1 * y,
                     f1 * y - f2 * z - 0.4 * y,
                     f2 * z - 0.01 * z], axis=-1)

def rk4(states, h):
    k1 = rhs(states)
    k2 = rhs(states + h * k1 / 2)
    k3 = rhs(states + h * k2 / 2)
    k4 = rhs(states + h * k3)
    return states + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6

def run(h=0.05, end=30000, epsilon=1e-7, renorm=1.0):
    n = round(end / h)
    every = round(renorm / h)
    sample = round(0.5 / h)
    assert np.isclose(every * h, renorm) and np.isclose(sample * h, 0.5)
    states = np.tile([0.5, 0.1, 0.1], (2, 2, 1)).astype(float)
    states[:, 1] += scales * epsilon / np.sqrt(3)
    minimum = states.copy()
    trajectory = np.empty((n // sample + 1, 2, 3))
    trajectory[0] = states[:, 0]
    logs = np.empty((n // every, 2))
    for i in range(1, n + 1):
        states = rk4(states, h)
        minimum = np.minimum(minimum, states)
        if i % every == 0:
            delta = states[:, 1] - states[:, 0]
            rho = np.linalg.norm(delta / scales, axis=1)
            if np.any(rho <= 0) or not np.all(np.isfinite(rho)):
                raise ValueError('Нулевое или нечисловое расстояние: проверьте epsilon и шаг.')
            logs[i // every - 1] = np.log(rho / epsilon)
            states[:, 1] = states[:, 0] + delta * (epsilon / rho)[:, None]
            minimum = np.minimum(minimum, states)
        if i % sample == 0:
            trajectory[i // sample] = states[:, 0]
    assert np.all(minimum >= 0) and np.all(np.isfinite(logs))
    return {'time': np.arange(len(trajectory)) * 0.5,
            'states': trajectory, 'logs': logs, 'minimum': minimum,
            'h': h, 'epsilon': epsilon, 'renorm': renorm}

def estimate(result, transient=10000, duration=20000):
    dt = float(result['renorm'])
    first = round(transient / dt)
    count = round(duration / dt)
    logs = result['logs'][first:first + count]
    if len(logs) != count:
        raise ValueError('Недостаточная длительность расчёта.')
    time = np.arange(1, count + 1) * dt
    return time, np.cumsum(logs, axis=0) / time[:, None]


import matplotlib.pyplot as plt
from io import BytesIO
from base64 import b64encode
from PIL import Image as PILImage
from IPython.display import display, HTML

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "font.size": 10,
                     "axes.grid": True, "grid.alpha": 0.25})

## 2. Расчёт траекторий

Расчёт проводится до $t=30000$ на шагах 0.1, 0.05 и 0.025.
Основной вариант: $h=0.05$, $t_{\mathrm{пер}}=10000$, $T=20000$.
Для построения графиков сохраняются состояния через 0.5 единицы времени;
интегрирование выполняется с указанным меньшим шагом.



In [2]:
runs = {h: run(h=h) for h in [0.1, 0.05, 0.025]}
main = runs[0.05]
transient = 10000
duration = 20000
print("h       b1      min x         min y         min z")
for h, result in runs.items():
    for j, b in enumerate(parameters):
        minimum = result["minimum"][j].min(axis=0)
        print(f"{h:<8g}{b:<8g}" + "  ".join(f"{v:.8f}" for v in minimum))

## 3. Временные ряды и фазовые портреты

Временные ряды показаны на общем участке $10000\leq t\leq11500$.
Фазовые портреты используют весь интервал $10000\leq t\leq30000$.
Пределы осей для одинаковых координат в двух режимах совпадают.
На графиках приведены сами популяции

In [3]:
t = main["time"]
states = main["states"]
tail = states[t >= transient]
lower = tail.min(axis=(0, 1))
upper = tail.max(axis=(0, 1))
margin = 0.06 * (upper - lower)
limits = list(zip(lower - margin, upper + margin))
names = ["x", "y", "z"]
colors = ["tab:blue", "tab:orange"]
mask = (t >= transient) & (t <= transient + 1500)

fig, axes = plt.subplots(3, 2, figsize=(12, 6.5), sharex=True, layout="constrained")
for j, b in enumerate(parameters):
    for k in range(3):
        axes[k, j].plot(t[mask], states[mask, j, k], color=colors[j], linewidth=0.8)
        axes[k, j].set(ylabel=names[k], ylim=limits[k])
    axes[0, j].set_title(f"b₁ = {b:g}")
    axes[-1, j].set_xlabel("t")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), layout="constrained")
for j, b in enumerate(parameters):
    for ax, (a, c) in zip(axes[j], [(0, 1), (0, 2), (1, 2)]):
        ax.plot(tail[:, j, a], tail[:, j, c], color=colors[j], linewidth=0.45)
        ax.set(xlabel=names[a], ylabel=names[c], xlim=limits[a], ylim=limits[c],
               title=f"b₁ = {b:g}")
plt.show()

## 4. Накопленная оценка показателя Ляпунова

На графиках $T$ отсчитывается от конца переходного процесса.
Для читаемости показаны значения начиная с $T=500$; в среднем учтены все
интервалы перенормировки после $t_{\mathrm{пер}}=10000$.
В таблице сравниваются конечные оценки на трёх шагах.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True, layout="constrained")
print("h       λ при b1=2.3      λ при b1=3")
for h, result in runs.items():
    average_time, values = estimate(result, transient, duration)
    print(f"{h:<8g}{values[-1, 0]: .7f}         {values[-1, 1]: .7f}")
    mask = average_time >= 500
    for j, ax in enumerate(axes):
        ax.plot(average_time[mask], values[mask, j], label=f"h = {h:g}", linewidth=0.9)
        ax.set(xlabel="Время усреднения T", ylabel="λmax(T)", title=f"b₁ = {parameters[j]:g}")
for ax in axes:
    ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
    ax.legend()
plt.show()

## 5. Переходный процесс, время усреднения и параметры расчёта

Сначала при фиксированном $T=20000$ меняется отброшенное время.
Затем при $t_{\mathrm{пер}}=10000$ меняется только время усреднения.
Для этого используются разные отрезки одной рассчитанной последовательности логарифмов.
Изменение начала окна меняет и его конец, поэтому различия включают конечновременные
колебания оценки, а не только влияние переходного процесса.

Отдельно повторяются расчёты с увеличенным возмущением и с удвоенным интервалом
перенормировки. Остальные настройки в каждой такой проверке сохранены.

In [5]:
print("Переходный процесс; h=0.05, T=20000")
print("t_пер      λ(b1=2.3)       λ(b1=3)")
for cut in [1000, 5000, 10000]:
    _, value = estimate(main, cut, 20000)
    print(f"{cut:<12}{value[-1, 0]: .7f}      {value[-1, 1]: .7f}")

print("\nВремя усреднения; h=0.05, t_пер=10000")
print("T          λ(b1=2.3)       λ(b1=3)")
for length in [5000, 10000, 20000]:
    _, value = estimate(main, 10000, length)
    print(f"{length:<12}{value[-1, 0]: .7f}      {value[-1, 1]: .7f}")

epsilon_check = run(h=0.05, epsilon=1e-6)
renorm_check = run(h=0.05, renorm=2.0)
print("\nПараметры перенормировки; h=0.05, t_пер=10000, T=20000")
print("ε          Δt_r      λ(b1=2.3)       λ(b1=3)")
for result in [main, epsilon_check, renorm_check]:
    _, value = estimate(result)
    print(f"{result['epsilon']:<11g}{result['renorm']:<10g}"
          f"{value[-1, 0]: .7f}      {value[-1, 1]: .7f}")

## 6. Проверка повторяемости колебаний

При $b_1=2.3$ внутри одного периода видны два максимума $z$ разной высоты.
Для оценки полного периода выбираются только большие максимумы, $z>12.6$.
При $b_1=3$ сравниваются интервалы между крупными максимумами $z>10$.
Положение каждого максимума уточняется параболой по трём соседним отсчётам с шагом 0.5.
Это дополнительная проверка периодичности: нулевой старший показатель сам по себе
не отличает цикл, например, от квазипериодического движения.

In [6]:
peak_statistics = []
print("b1      Число пиков    Средний интервал    Минимум     Максимум     СКО")
for j, threshold in enumerate([12.6, 10.0]):
    z = states[:, j, 2]
    peaks = np.flatnonzero((z[1:-1] > z[:-2]) & (z[1:-1] > z[2:])) + 1
    peaks = peaks[(t[peaks] >= transient) & (z[peaks] > threshold)]
    peak_time = t[peaks] + 0.25 * (z[peaks - 1] - z[peaks + 1]) / (
        z[peaks - 1] - 2 * z[peaks] + z[peaks + 1])
    intervals = np.diff(peak_time)
    peak_statistics.append([intervals.mean(), intervals.min(), intervals.max(), intervals.std()])
    print(f"{parameters[j]:<8g}{len(peaks):<14}{intervals.mean():.5f}"
          f"             {intervals.min():.5f}   {intervals.max():.5f}   {intervals.std():.5f}")

## 7. Анимация фазовой траектории и временных рядов

Показан интервал $10000\leq t\leq10500$ для обоих режимов с общим модельным временем.
Слева точка движется по трёхмерной фазовой траектории, справа отмечены соответствующие
значения $x,y,z$. Кадры взяты из рассчитанных узлов через 2 единицы времени;
воспроизведение ускорено. Пределы осей одинаковы для двух режимов.

In [7]:
frame_indices = np.flatnonzero((t >= transient) & (t <= transient + 500))[::4]
frame_times = t[frame_indices]
window = (t >= transient) & (t <= transient + 500)
fig = plt.figure(figsize=(11, 7.5), dpi=90, layout="constrained")
grid = fig.add_gridspec(6, 2, width_ratios=[1, 1.6])
phase_points = []
series_points = []
cursors = []
for j, b in enumerate(parameters):
    ax = fig.add_subplot(grid[3*j:3*j+3, 0], projection="3d")
    q = states[window, j]
    ax.plot(q[:, 0], q[:, 1], q[:, 2], color=colors[j], linewidth=0.7, alpha=0.6)
    point, = ax.plot([], [], [], "o", color=colors[j], markersize=6)
    phase_points.append(point)
    ax.set(xlabel="x", ylabel="y", zlabel="z", title=f"b₁ = {b:g}",
           xlim=limits[0], ylim=limits[1], zlim=limits[2])
    ax.set_box_aspect((1, 1, 1), zoom=0.78)
    ax.set_xticks([0.2, 0.6, 1.0])
    ax.set_yticks([0.0, 0.2, 0.4])
    ax.set_zticks([8, 10, 12])
    ax.tick_params(labelsize=8)
    ax.view_init(elev=25, azim=-55)
    group_points, group_cursors = [], []
    for k in range(3):
        curve = fig.add_subplot(grid[3*j+k, 1])
        curve.plot(t[window], states[window, j, k], color=colors[j], linewidth=0.9)
        dot, = curve.plot([], [], "o", color=colors[j], markersize=4)
        cursor = curve.axvline(transient, color="black", linestyle="--", linewidth=0.8)
        curve.set(ylabel=names[k], ylim=limits[k], xlim=(transient, transient+500))
        if k == 2:
            curve.set_xlabel("t")
        else:
            curve.tick_params(labelbottom=False)
        group_points.append(dot)
        group_cursors.append(cursor)
    series_points.append(group_points)
    cursors.append(group_cursors)
title = fig.suptitle("")


def update_frame(frame):
    index = frame_indices[frame]
    now = t[index]
    for j in range(2):
        x, y, z = states[index, j]
        phase_points[j].set_data([x], [y])
        phase_points[j].set_3d_properties([z])
        for k in range(3):
            series_points[j][k].set_data([now], [states[index, j, k]])
            cursors[j][k].set_xdata([now, now])
    title.set_text(f"t = {now:.0f}")


frames = []
for i in range(len(frame_indices)):
    update_frame(i)
    fig.canvas.draw()
    frames.append(PILImage.fromarray(np.asarray(fig.canvas.buffer_rgba())).convert("RGB"))
buffer = BytesIO()
frames[0].save(buffer, format="GIF", save_all=True, append_images=frames[1:], duration=50, loop=0)
plt.close(fig)
gif_data = b64encode(buffer.getvalue()).decode("ascii")
display(HTML(f'<img alt="Пищевая цепь: два режима" src="data:image/gif;base64,{gif_data}">'))

## Выводы

При $b_1=2.3$ фазовая траектория замыкается, а полный период колебаний составляет
примерно 122.774. Разброс уточнённых интервалов около $10^{-3}$ связан с отсчётами
и интерполяцией максимумов. Оценка старшего показателя при $t_{\mathrm{пер}}=10000$,
$T=20000$, $h=0.05$ равна примерно $-4.8\cdot10^{-5}$.
При изменении времени усреднения она меняет знак и остаётся близкой к нулю.
Это согласуется с устойчивым предельным циклом автономной системы: возмущение вдоль
цикла соответствует сдвигу фазы и имеет нулевой показатель, а поперечные возмущения затухают.
Поперечные показатели здесь отдельно не вычислялись.

При $b_1=3$ траектория не замыкается в одну кривую, а интервалы между крупными
максимумами меняются примерно от 114.70 до 137.97.
Для трёх шагов получены положительные оценки $0.00640$, $0.00701$ и $0.00663$.
Увеличение начального возмущения с $10^{-7}$ до $10^{-6}$ и изменение интервала
перенормировки с 1 до 2 практически не изменили оценку при $h=0.05$.
Поэтому результат следует записывать как величину порядка $7\cdot10^{-3}$,
а не считать все цифры в таблице точными. Различия между длинными расчётами
связаны также с конечным временем усреднения и разными участками хаотической траектории.

Количественное отличие состоит в устойчиво положительном старшем показателе
для ограниченного хаотического движения и нулевом — для предельного цикла.
Один малый положительный результат на коротком интервале недостаточен.
Нужны проверка переходного процесса, увеличение времени усреднения и контроль
численного метода. В выполненных проверках знак и порядок величины показателя
при $b_1=3$ сохраняются; при $b_1=2.3$ данные согласуются с нулём и повторяемым циклом.